# Taller Integrador — Sesión 14
# Pruebas de Software
# QA Lead - Analítica Académica
# Nombre: Johan
# Fecha: Abril 2026

In [11]:
def detectar_estudiantes_riesgo(notas: list, umbral: float = 3.0) -> list:
    if not notas:
        return []

    riesgo = []

    for estudiante in notas:
        if not estudiante.get('materias'):
            continue

        suma = sum(m['nota'] for m in estudiante['materias'])
        promedio = suma / len(estudiante['materias'])

        if promedio < umbral:
            riesgo.append({'id': estudiante['id'], 'promedio': promedio})

    return riesgo

In [12]:
def test_lista_vacia():
    assert detectar_estudiantes_riesgo([]) == []

def test_sin_materias():
    data = [{'id': 1, 'materias': []}]
    assert detectar_estudiantes_riesgo(data) == []

def test_en_riesgo():
    data = [{'id': 1, 'materias': [{'nota': 2.0}, {'nota': 2.5}]}]
    assert len(detectar_estudiantes_riesgo(data)) == 1

def test_aprobado():
    data = [{'id': 1, 'materias': [{'nota': 4.0}, {'nota': 3.5}]}]
    assert detectar_estudiantes_riesgo(data) == []

def test_umbral():
    data = [{'id': 1, 'materias': [{'nota': 3.0}, {'nota': 3.0}]}]
    assert len(detectar_estudiantes_riesgo(data, 3.5)) == 1

In [13]:
test_lista_vacia()
test_sin_materias()
test_en_riesgo()
test_aprobado()
test_umbral()

print("✅ Todos los tests pasaron")

✅ Todos los tests pasaron


In [14]:
!pip install hypothesis

In [15]:
from hypothesis import given, strategies as st

@given(
    st.lists(
        st.fixed_dictionaries({
            "id": st.integers(min_value=1, max_value=1000),
            "materias": st.lists(
                st.fixed_dictionaries({
                    "nota": st.floats(min_value=0.0, max_value=5.0)
                }),
                min_size=0  # Allow empty list of materias, as the function handles it
            )
        })
    )
)
def test_propiedad(estudiantes):
    result = detectar_estudiantes_riesgo(estudiantes, umbral=10)

    for r in result:
        assert r['promedio'] <= 5

test_propiedad()
print("✅ Test de propiedad OK")

✅ Test de propiedad OK


In [16]:
def seleccionar_tests_optimos(tests, presupuesto):
    n = len(tests)
    dp = [[0]*(presupuesto+1) for _ in range(n+1)]

    for i in range(1, n+1):
        nombre, tiempo, valor = tests[i-1]
        for w in range(presupuesto+1):
            if tiempo <= w:
                dp[i][w] = max(dp[i-1][w], dp[i-1][w-tiempo] + valor)
            else:
                dp[i][w] = dp[i-1][w]

    w = presupuesto
    seleccionados = []

    for i in range(n, 0, -1):
        if dp[i][w] != dp[i-1][w]:
            nombre, tiempo, valor = tests[i-1]
            seleccionados.append(nombre)
            w -= tiempo

    return dp[n][presupuesto], seleccionados[::-1]

In [17]:
tests = [
    ("test_login",3,85),
    ("test_calculo_promedio",5,95),
    ("test_deteccion_riesgo",4,90),
    ("test_exportar_pdf",6,70),
    ("test_exportar_excel",5,75),
    ("test_api_reportes",3,80),
    ("test_prediccion",8,65),
    ("test_ui_dashboard",7,60),
    ("test_concurrency",9,55),
    ("test_seguridad",4,88)
]

In [18]:
valor, seleccion = seleccionar_tests_optimos(tests, 25)

print("Valor total:", valor)
print("Tests seleccionados:", seleccion)

Valor total: 513
Tests seleccionados: ['test_login', 'test_calculo_promedio', 'test_deteccion_riesgo', 'test_exportar_excel', 'test_api_reportes', 'test_seguridad']
